In [ ]:
import pandas as pd
import seaborn as sns  
import matplotlib.pyplot as plt
import plotly.express as px
import hashlib
import numpy as np
import missingno as msno
import os
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

In [ ]:
pd.read_excel("Data/Diccionario_Variables_Alumnos.xlsx")

In [ ]:
dic=pd.read_excel("Data/Diccionario_Variables_Alumnos.xlsx")
dic

In [ ]:
df=pd.read_excel("Data/Prestamos_Data_Alumnos_v5.xlsx")
df

In [ ]:
df1 = df[df["Proposito"] == "Negocios"].reset_index(drop=True)
df1 = df1.drop(columns=["Proposito"])

In [ ]:
filtrado = df1.copy()

In [ ]:
filtrado.info()

In [ ]:
filtrado.describe()

In [ ]:
filtrado.shape

In [ ]:
filtrado.columns.unique()

In [ ]:
filtrado.isna().sum().sum()

In [ ]:
null = filtrado[filtrado.isna().any(axis=1)]
null

In [ ]:
filtrado = filtrado.dropna(axis=0, how='any')

In [ ]:
filtrado.duplicated().sum()

In [ ]:
filtrado = filtrado.drop_duplicates()

In [ ]:
filtrado

In [ ]:
df.shape[0]

In [ ]:
filtrado.shape[0]

In [ ]:
total = filtrado.shape[0]/df.shape[0]
total

In [ ]:
# 1. Definir el encoder indicando que queremos un array denso directamente
onehot_enc = OneHotEncoder(sparse_output=False)


# 2. Ajustar y transformar las variables nominales
columnas_categoricas = ["Estado_Civil", "Tipo_Jornada_Laboral"]
datos_onehot = onehot_enc.fit_transform(filtrado[columnas_categoricas])


# 3. Obtener los nombres de las nuevas columnas
feature_names = onehot_enc.get_feature_names_out(input_features=columnas_categoricas)


# 4. Creacion de un DataFrame con los datos codificados y usamos la columna "ID" para unirlo con el DataFrame original
data_encoded = pd.DataFrame(datos_onehot, columns=feature_names)
data_encoded["ID"] = filtrado["ID"].values


# 5. Unimos ambos DataFrames usando pd.merge() en base a la columna "ID"
datos = pd.merge(filtrado.drop(columns=columnas_categoricas), data_encoded, on="ID")


In [ ]:
data_encoded

In [ ]:
datos

In [ ]:
filtrado["Estudios"].unique()

In [ ]:
# 1. Definimos el orden de menor a mayor
orden_estudios = [["Escolar", "Grado Universitario", "Doctorado", "Máster"]]


# 2. Creamos el modelo y usamos las columnas que queremos darle orden
ordinal_enc = OrdinalEncoder(categories=orden_estudios)


# 3. Ajustamos y transformamos la columna (ademas, se sumamos 1 para que el valor minimo sean 1 y no 0, le convertimos a int)
datos["Estudios"] = (ordinal_enc.fit_transform(datos[["Estudios"]]) + 1).astype(int)

In [ ]:
filtrado

In [ ]:
datos

In [ ]:
datos.info()

In [ ]:
df.columns

In [ ]:
df["Interes_Mensual"] = (df["Ratio_Interes"] / 100) / 12

In [ ]:
df[["Ratio_Interes", "Interes_Mensual"]].head()

In [ ]:
df["Cuota_Francesa"] = (
    df["Monto_Inicial"] *
    (
        df["Interes_Mensual"] * (1 + df["Interes_Mensual"]) ** df["Duracion"]
    ) /
    (
        (1 + df["Interes_Mensual"]) ** df["Duracion"] - 1
    )
)

In [ ]:
df[["Monto_Inicial", "Duracion", "Cuota_Francesa"]].head()

In [ ]:
prestamo = df.iloc[0]

In [ ]:
capital = prestamo["Monto_Inicial"]
interes = prestamo["Interes_Mensual"]
meses = int(prestamo["Duracion"])
cuota = prestamo["Cuota_Francesa"]

In [ ]:
capital_pendiente = capital

tabla = []

for mes in range(1, meses + 1):

    interes_mes = capital_pendiente * interes

    amortizacion = cuota - interes_mes

    capital_pendiente = capital_pendiente - amortizacion

    tabla.append([
        mes,
        cuota,
        interes_mes,
        amortizacion,
        max(capital_pendiente, 0)
    ])

In [ ]:
tabla_frances = pd.DataFrame(
    tabla,
    columns=[
        "Mes",
        "Cuota",
        "Interes",
        "Amortizacion",
        "Capital_Pendiente"
    ]
)

In [ ]:
tabla_frances.head()

In [ ]:
tabla_frances.tail()

In [ ]:
def cuadro_frances(capital, interes_mensual, meses):

    cuota = capital * (
        interes_mensual * (1 + interes_mensual) ** meses
    ) / (
        (1 + interes_mensual) ** meses - 1
    )

    capital_pendiente = capital

    tabla = []

    for mes in range(1, meses + 1):

        interes = capital_pendiente * interes_mensual

        amortizacion = cuota - interes

        capital_pendiente -= amortizacion

        tabla.append({
            "Mes": mes,
            "Cuota": cuota,
            "Interes": interes,
            "Amortizacion": amortizacion,
            "Capital_Pendiente": max(capital_pendiente, 0)
        })

    return pd.DataFrame(tabla)

In [ ]:
tabla_frances = cuadro_frances(
    df.loc[0, "Monto_Inicial"],
    df.loc[0, "Interes_Mensual"],
    int(df.loc[0, "Duracion"])
)

tabla_frances.head()

In [ ]:
primera_cuota = tabla_frances.loc[0, "Cuota"]
intereses_totales = tabla_frances["Interes"].sum()
capital_final = tabla_frances.iloc[-1]["Capital_Pendiente"]

In [ ]:
def resumen_frances(capital, interes_mensual, meses):

    cuota = capital * (
        interes_mensual * (1 + interes_mensual) ** meses
    ) / (
        (1 + interes_mensual) ** meses - 1
    )

    capital_pendiente = capital
    intereses_totales = 0

    for mes in range(1, meses + 1):

        interes = capital_pendiente * interes_mensual

        amortizacion = cuota - interes

        capital_pendiente -= amortizacion

        intereses_totales += interes

        if mes == meses // 2:
            capital_mitad = capital_pendiente

    return cuota, intereses_totales, capital_mitad

In [ ]:
df_frances = df.dropna(subset=["Monto_Inicial", "Interes_Mensual", "Duracion"]).copy()

In [ ]:
resultados = df_frances.apply(
    lambda fila: resumen_frances(
        fila["Monto_Inicial"],
        fila["Interes_Mensual"],
        int(fila["Duracion"])
    ),
    axis=1
)

In [ ]:
df_frances["Cuota_Francesa"] = resultados.apply(lambda x: x[0])

df_frances["Intereses_Totales"] = resultados.apply(lambda x: x[1])

df_frances["Capital_Mitad"] = resultados.apply(lambda x: x[2])

In [ ]:
df_frances[
    [
        "Monto_Inicial",
        "Duracion",
        "Cuota_Francesa",
        "Intereses_Totales",
        "Capital_Mitad"
    ]
].head()

In [ ]:
df_frances[
    [
        "Cuota_Francesa",
        "Intereses_Totales",
        "Capital_Mitad"
    ]
].isna().sum()

In [ ]:
df_frances["Cuotas_Restantes"] = (
    df_frances["Duracion"] - (df_frances["Duracion"] // 2)
)

In [ ]:
df_frances["Valor_Presente"] = (
    df_frances["Cuota_Francesa"]
    * (
        1 - (1 + df_frances["Interes_Mensual"]) ** (-df_frances["Cuotas_Restantes"])
    )
    / df_frances["Interes_Mensual"]
)

In [ ]:
df_frances[
    [
        "Monto_Inicial",
        "Cuota_Francesa",
        "Capital_Mitad",
        "Cuotas_Restantes",
        "Valor_Presente"
    ]
].head()